# EDA — RSNA 2020 Pulmonary Embolism Detection (Grupo 2)

Análise exploratória inicial (Marco 1, Semana 1): distribuição de classes, quantidade de exames/pacientes, metadados DICOM relevantes e exemplos visuais por classe.

**Sobre os dados usados aqui:** por padrão este notebook aponta para `data/sample/synthetic_manifest.json`, gerado por `scripts/generate_synthetic_sample.py`, porque o ambiente de preparação deste repositório não tem acesso ao Kaggle. Assim que a amostra real do desafio estiver em `data/raw/` (ver `README.md`), troque `MANIFEST_PATH`/`DATA_ROOT` abaixo pelo manifesto real (derivado de `train.csv`) e reexecute — todo o resto do notebook é agnóstico à origem dos dados.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

from src.config import DATA_SAMPLE_DIR
from src.data.dicom_io import load_exam_slices
from src.preprocessing.pipeline import preprocess_slice

MANIFEST_PATH = DATA_SAMPLE_DIR / "synthetic_manifest.json"
DATA_ROOT = ROOT_DIR

assert MANIFEST_PATH.exists(), "Rode antes: python scripts/generate_synthetic_sample.py"
records = json.loads(MANIFEST_PATH.read_text())
df = pd.DataFrame(records)
df.head()

## 1. Distribuição de classes e de exames por paciente

In [ ]:
print("Exames:", len(df))
print("Pacientes únicos:", df["patient_id"].nunique())
print("Exames por paciente (max):", df.groupby("patient_id").size().max())
print("\nDistribuição de classes (pe_present_on_exam):")
print(df["pe_present_on_exam"].value_counts())
print("Proporção de positivos: {:.1%}".format(df["pe_present_on_exam"].mean()))

df["pe_present_on_exam"].value_counts().plot(kind="bar", title="Distribuição de classes (nível de exame)")
plt.xlabel("pe_present_on_exam")
plt.ylabel("n. de exames")
plt.show()

## 2. Metadados DICOM relevantes (modalidade, espaçamento de pixel, RescaleSlope/Intercept)

Com dados reais, esta seção deve reportar a variabilidade de `PixelSpacing`, `SliceThickness`, fabricante/modelo do scanner e protocolo — parte da justificativa da reamostragem isotrópica (§4.1 do enunciado).

In [ ]:
meta_rows = []
for record in records[:8]:
    exam_dir = DATA_ROOT / record["exam_dir"]
    slices = load_exam_slices(exam_dir)
    if not slices:
        continue
    s = slices[0]
    meta_rows.append(
        {
            "patient_id": record["patient_id"],
            "modality": s.modality,
            "pixel_spacing": s.pixel_spacing,
            "slice_thickness": s.slice_thickness,
            "rescale_slope": s.rescale_slope,
            "rescale_intercept": s.rescale_intercept,
            "n_slices": len(slices),
        }
    )
pd.DataFrame(meta_rows)

## 3. Exemplos visuais por classe (corte central, após janelamento)

In [ ]:
def show_example(record, ax):
    exam_dir = DATA_ROOT / record["exam_dir"]
    slices = load_exam_slices(exam_dir)
    mid = slices[len(slices) // 2]
    image = preprocess_slice(mid)
    ax.imshow(image, cmap="gray")
    ax.set_title(f"paciente={record['patient_id']} | label={record['pe_present_on_exam']}")
    ax.axis("off")


positive_example = next(r for r in records if r["pe_present_on_exam"] == 1)
negative_example = next(r for r in records if r["pe_present_on_exam"] == 0)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
show_example(positive_example, axes[0])
show_example(negative_example, axes[1])
plt.tight_layout()
plt.show()

## 4. Formulação da tarefa (esboço — Marco 1)

- **Entrada:** um exame de angio-TC de tórax (série de cortes axiais em DICOM), lido e convertido para HU.
- **Saída (baseline):** classificação binária no nível do EXAME — presença (1) ou ausência (0) de embolia pulmonar, correspondente ao inverso de `negative_exam_for_pe` no `train.csv` original do desafio.
- **Unidade de amostra:** o EXAME (`StudyInstanceUID`), agregando características extraídas por corte via pooling estatístico (média/desvio/percentil 90/máximo) — ver `src/features/aggregate.py` para a justificativa completa dessa decisão.
- **Unidade de partição:** o PACIENTE (`PatientID`), via `StratifiedGroupKFold`, para impedir vazamento entre treino e teste.

Próximos passos (Semana 2): confirmar esta formulação contra a descrição oficial do desafio (Colak et al., 2021) e o dicionário de dados do Kaggle; refinar a estratégia de agregação se a literatura sugerir alternativa melhor sustentada.